In [1]:
import pandas as pd
import math
import matplotlib.pyplot as plt

df = pd.read_csv("../data/ab_test_data.csv")

print(df.head())
print()
print(df.shape)

   user_id      group   device traffic_source  converted
0        1    control  desktop           paid          0
1        2  treatment   mobile           paid          0
2        3  treatment   mobile           paid          0
3        4  treatment  desktop           paid          0
4        5    control   mobile           paid          0

(50000, 5)


In [2]:
combined_analysis = (
    df.groupby(["device", "traffic_source", "group"])
    .agg(
        users=("user_id", "count"),
        conversions=("converted", "sum"),
        conversion_rate=("converted", "mean")
    )
    .reset_index()
)

combined_analysis["conversion_rate_pct"] = (
    combined_analysis["conversion_rate"] * 100
)

print(combined_analysis.round(4))

     device traffic_source      group  users  conversions  conversion_rate  \
0   desktop          email    control   1640          225           0.1372   
1   desktop          email  treatment   1598          251           0.1571   
2   desktop        organic    control   2863          421           0.1470   
3   desktop        organic  treatment   2658          413           0.1554   
4   desktop           paid    control   2437          355           0.1457   
5   desktop           paid  treatment   2429          357           0.1470   
6   desktop         social    control   1306          179           0.1371   
7   desktop         social  treatment   1241          193           0.1555   
8    mobile          email    control   2981          320           0.1073   
9    mobile          email  treatment   3083          352           0.1142   
10   mobile        organic    control   5152          544           0.1056   
11   mobile        organic  treatment   5268          554       

In [3]:
combined_pivot = combined_analysis.pivot_table(
    index=["device", "traffic_source"],
    columns="group",
    values="conversion_rate"
)

combined_pivot["absolute_lift"] = (
    combined_pivot["treatment"] - combined_pivot["control"]
)

combined_pivot["relative_lift"] = (
    combined_pivot["absolute_lift"] / combined_pivot["control"]
)

combined_pivot["control_pct"] = combined_pivot["control"] * 100
combined_pivot["treatment_pct"] = combined_pivot["treatment"] * 100
combined_pivot["lift_pp"] = combined_pivot["absolute_lift"] * 100
combined_pivot["relative_lift_pct"] = combined_pivot["relative_lift"] * 100

segment_lift = combined_pivot[
    [
        "control_pct",
        "treatment_pct",
        "lift_pp",
        "relative_lift_pct"
    ]
].sort_values("lift_pp", ascending=False)

print(segment_lift.round(2))

group                   control_pct  treatment_pct  lift_pp  relative_lift_pct
device  traffic_source                                                        
tablet  social                 9.12          14.20     5.07              55.58
        email                 10.16          12.84     2.69              26.44
desktop email                 13.72          15.71     1.99              14.49
        social                13.71          15.55     1.85              13.47
mobile  paid                  10.22          11.27     1.05              10.27
desktop organic               14.70          15.54     0.83               5.67
mobile  email                 10.73          11.42     0.68               6.36
        social                10.68          10.82     0.14               1.33
desktop paid                  14.57          14.70     0.13               0.89
mobile  organic               10.56          10.52    -0.04              -0.40
tablet  paid                  11.58          10.63  

In [4]:
segment_sizes = (
    combined_analysis
    .pivot_table(
        index=["device", "traffic_source"],
        columns="group",
        values="users"
    )
)

segment_summary = segment_lift.copy()

segment_summary["control_users"] = segment_sizes["control"]
segment_summary["treatment_users"] = segment_sizes["treatment"]

segment_summary = segment_summary[
    [
        "control_users",
        "treatment_users",
        "control_pct",
        "treatment_pct",
        "lift_pp",
        "relative_lift_pct"
    ]
]

print(segment_summary.round(2))

group                   control_users  treatment_users  control_pct  \
device  traffic_source                                                
tablet  social                  274.0            317.0         9.12   
        email                   384.0            366.0        10.16   
desktop email                  1640.0           1598.0        13.72   
        social                 1306.0           1241.0        13.71   
mobile  paid                   4481.0           4516.0        10.22   
desktop organic                2863.0           2658.0        14.70   
mobile  email                  2981.0           3083.0        10.73   
        social                 2210.0           2255.0        10.68   
desktop paid                   2437.0           2429.0        14.57   
mobile  organic                5152.0           5268.0        10.56   
tablet  paid                    587.0            574.0        11.58   
        organic                 738.0            642.0        12.20   

group

In [5]:
combined_results = []

for (device, source), segment in df.groupby(["device", "traffic_source"]):

    control = segment[segment["group"] == "control"]["converted"]
    treatment = segment[segment["group"] == "treatment"]["converted"]

    control_users = len(control)
    treatment_users = len(treatment)

    control_conversions = control.sum()
    treatment_conversions = treatment.sum()

    control_rate = control.mean()
    treatment_rate = treatment.mean()

    pooled_rate = (
        control_conversions + treatment_conversions
    ) / (
        control_users + treatment_users
    )

    standard_error = math.sqrt(
        pooled_rate * (1 - pooled_rate) *
        ((1 / control_users) + (1 / treatment_users))
    )

    z_stat = (control_rate - treatment_rate) / standard_error

    p_value = 0.5 * (
        1 + math.erf(z_stat / math.sqrt(2))
    )

    combined_results.append({
        "device": device,
        "traffic_source": source,
        "control_users": control_users,
        "treatment_users": treatment_users,
        "lift_pp": (treatment_rate - control_rate) * 100,
        "p_value": p_value,
        "significant": p_value < 0.05
    })

combined_test_results = pd.DataFrame(combined_results)

combined_test_results = combined_test_results.sort_values(
    "p_value"
)

print(combined_test_results.round(4))

     device traffic_source  control_users  treatment_users  lift_pp  p_value  \
11   tablet         social            274              317   5.0715   0.0285   
6    mobile           paid           4481             4516   1.0501   0.0539   
0   desktop          email           1640             1598   1.9876   0.0551   
3   desktop         social           1306             1241   1.8460   0.0937   
8    tablet          email            384              366   2.6853   0.1243   
1   desktop        organic           2863             2658   0.8331   0.1939   
4    mobile          email           2981             3083   0.6828   0.1986   
7    mobile         social           2210             2255   0.1417   0.4393   
2   desktop           paid           2437             2429   0.1303   0.4488   
5    mobile        organic           5152             5268  -0.0427   0.5283   
10   tablet           paid            587              574  -0.9571   0.6981   
9    tablet        organic            73

In [6]:
number_of_tests = len(combined_test_results)

bonferroni_alpha = 0.05 / number_of_tests

combined_test_results["bonferroni_significant"] = (
    combined_test_results["p_value"] < bonferroni_alpha
)

print(f"Number of Tests: {number_of_tests}")
print(f"Original Alpha: 0.0500")
print(f"Bonferroni Adjusted Alpha: {bonferroni_alpha:.4f}")
print()

print(
    combined_test_results[
        [
            "device",
            "traffic_source",
            "lift_pp",
            "p_value",
            "bonferroni_significant"
        ]
    ].round(4)
)

Number of Tests: 12
Original Alpha: 0.0500
Bonferroni Adjusted Alpha: 0.0042

     device traffic_source  lift_pp  p_value  bonferroni_significant
11   tablet         social   5.0715   0.0285                   False
6    mobile           paid   1.0501   0.0539                   False
0   desktop          email   1.9876   0.0551                   False
3   desktop         social   1.8460   0.0937                   False
8    tablet          email   2.6853   0.1243                   False
1   desktop        organic   0.8331   0.1939                   False
4    mobile          email   0.6828   0.1986                   False
7    mobile         social   0.1417   0.4393                   False
2   desktop           paid   0.1303   0.4488                   False
5    mobile        organic  -0.0427   0.5283                   False
10   tablet           paid  -0.9571   0.6981                   False
9    tablet        organic  -1.1359   0.7440                   False


## Multiple Testing Correction

The combined device and traffic-source analysis tested 12 different customer segments.

Before adjustment, the Tablet + Social segment appeared statistically significant with a p-value of 0.0285. However, testing many segments increases the probability of finding a significant result by chance.

Using the Bonferroni correction, the significance threshold was adjusted from 0.05 to approximately 0.0042.

After applying this correction, none of the combined segments remained statistically significant.

Therefore, the apparent Tablet + Social effect should be treated as exploratory rather than strong evidence of a true treatment effect.

In [8]:
overall_summary = (
    df.groupby("group")
    .agg(
        users=("user_id", "count"),
        conversions=("converted", "sum"),
        conversion_rate=("converted", "mean")
    )
)

control_rate = overall_summary.loc["control", "conversion_rate"]
treatment_rate = overall_summary.loc["treatment", "conversion_rate"]

absolute_lift = treatment_rate - control_rate
relative_lift = absolute_lift / control_rate

summary = pd.DataFrame({
    "Metric": [
        "Control Users",
        "Treatment Users",
        "Control Conversion Rate",
        "Treatment Conversion Rate",
        "Absolute Lift",
        "Relative Lift"
    ],
    "Value": [
        f"{overall_summary.loc['control', 'users']:,}",
        f"{overall_summary.loc['treatment', 'users']:,}",
        f"{control_rate:.2%}",
        f"{treatment_rate:.2%}",
        f"{absolute_lift:.2%}",
        f"{relative_lift:.2%}"
    ]
})

print(summary.to_string(index=False))

                   Metric  Value
            Control Users 25,053
          Treatment Users 24,947
  Control Conversion Rate 11.81%
Treatment Conversion Rate 12.41%
            Absolute Lift  0.60%
            Relative Lift  5.07%
